> **구분: 수업 후 복습·자습**

# 자습 2. 포인터와 동적 메모리 — 약 60분

값이 저장된 **주소** , 주소를 담는 **포인터** , 기존 객체의 별칭인 **참조** , 실행 중 확보하는 동적 메모리를 차례로 연결합니다. 위험한 오류는 실행하지 않고 원인을 설명하며, 안전한 코드는 sanitizer로 검증합니다.

> 이번 교시의 목적은 `new/delete`를 많이 쓰는 것이 아니라, 수명과 소유권 문제를 이해해 현재 절의 RAII 권장 방식과 `std::vector`가 왜 필요한지 설명하는 것입니다.

<!-- standalone-review:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** Python 3 커널에서 첫 환경 셀의 `/opt/course/bin/python` 경로를 확인합니다. 코드 셀은 Shift+Enter로 실행하고 `[*]`가 끝난 뒤 다음 셀로 이동합니다. TODO 파일을 저장한 뒤 검증 셀을 실행합니다.

### 이 노트북만으로 시작하기

열 파일은 `/workspace/03.C++/91_selfstudy_memory.ipynb`이며 선택 자습입니다. [03_memory_stl_cmake.ipynb의 「주소·참조·포인터와 수명」](03_memory_stl_cmake.ipynb)에 개요가 있으며 필요한 내용은 `&x`가 주소, `*p`가 그 주소의 값, `nullptr`가 유효한 대상을 가리키지 않는 값이라는 점입니다. null을 역참조하지 않고 배열 주소와 길이를 함께 전달합니다. `new[]`로 확보한 배열은 소유자가 `delete[]`로 한 번 해제하며 빌려 읽는 함수는 해제하지 않습니다.

이전 코드·산출물은 필요하지 않습니다. 첫 환경 셀이 `/workspace` 작업 위치와 빌드 도우미를 준비합니다. `%%writefile` 작성 셀을 수정·실행해 `/workspace/03.C++/04_pointers_memory/exercise/main.cpp`에 저장하고 sanitizer 빌드·학습자 검증을 실행하세요. 외부 파일이 최신이면 먼저 작성 셀에 옮깁니다. 빌드 경로는 `/workspace/Data/build/03.C++/04_pointers_memory/`입니다.
<!-- standalone-review:end -->


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

이 선택 자습은 주소를 계산하는 요령보다 누가 메모리를 소유하고 언제까지 사용할 수 있는지 이해하는 과정입니다. 값이 출력된다는 사실만으로 잘못된 접근이 없다고 할 수 없으므로 함수 계약 검사와 sanitizer를 함께 사용합니다.

### 전체 과정에서 내 코드가 하는 일

```text
포인터·길이·수명 확인
  ↓
합계·평균·최솟값과 복사 TODO
  ↓
sanitizer 빌드
  ↓
경계 입력 검사
  ↓
최초 오류 수정
```

nullptr·빈 입력·일반 입력과 복사 후 독립성을 구현합니다. 원본이 바뀌었을 때 복사본이 함께 바뀌면 어떤 메모리를 공유하는지 설명합니다. 경고를 무시하지 않고 진단이 발생하면 실패로 끝나게 검사합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 평균 20.2가 출력되었지만 runtime error도 보이면 완료인가요?

**설명:** 아닙니다. 값의 정답과 정의된 메모리·연산 동작은 별도 조건입니다. 첫 진단 위치의 인덱스·길이·수명·연산 범위를 확인하고 수정 후 다시 빌드합니다. sanitizer가 중단 없이 통과하고 계약 검사도 통과해야 완료입니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| AddressSanitizer/UBSan 오류·segmentation fault | 첫 진단의 소스 줄, 배열 길이·인덱스·해제 후 사용·중복 해제를 확인합니다. sanitizer를 끄지 말고 소유권과 수명을 수정합니다. | sanitizer 진단 없이 메모리 계약 통과 |
| 평균/최솟값·빈 입력 검사 실패 | 배열 길이 전달과 빈 입력 분기를 정답과 비교합니다. | 일반·빈 입력 경계 검사 PASS |
| 수정한 결과가 반영되지 않음 | 본인 exercise 소스를 저장하고 빌드 셀부터 다시 실행합니다. 이전 실행 파일을 실행했는지 확인합니다. | 새 빌드의 학습자 검사 PASS |

**예시에 없는 오류를 확인할 곳:** [C++ 공통 안내서](BEGINNER_GUIDE.md), 해당 lesson의 tests와 빌드 stderr. CMake는 build 폴더의 `Testing/Temporary/LastTest.log`도 확인합니다. [CTest 공식 안내](https://cmake.org/cmake/help/latest/manual/ctest.1.html), [sanitizer 옵션](https://gcc.gnu.org/onlinedocs/gcc/Instrumentation-Options.html).


[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 후 복습·자습**

이 노트북은 수업 후 자습입니다. 본수업 필수 완료 조건에 포함하지 않습니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 제공된 빌드 틀에서 지정한 TODO를 수정하고 검증합니다.
- **단원 마지막 연결 질문:** Python 결과와 같음을 먼저 확인합니다. 기존 C++ 부품 재사용, 호출 비용, 수정·배포 부담 중 이 기능을 .so로 분리할 이유를 하나 선택합니다. C++이라는 이유만으로 빨라진다고 결론내리지 않습니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.


> **구분: 수업 후 복습·자습**

## 처음 학습하는 학생을 위한 노트북 사용법

이 노트북의 설명은 위에서 아래로 읽되, 모든 문법을 외운 뒤 코드를 실행하려고 하지 않아도 됩니다. **코드 읽기 → 결과 예상 → 셀 실행 → 실제 결과 비교 → TODO 수정 → 검증** 순서로 반복하면 됩니다.

| 셀 또는 파일 | 하는 일 | 학생이 할 일 |
|---|---|---|
| Markdown 설명 | 새 용어와 코드의 목적 설명 | 낯선 용어를 표시하고 실행 결과를 먼저 예상 |
| 환경 Python 셀 | `g++`·CMake를 호출할 도우미와 경로 준비 | 내용을 바꾸지 말고 먼저 실행 |
| `examples/` | 완성된 작은 C++ 예제 | 출력 전에 변수 변화를 손으로 추적 |
| `exercise/` | 학생이 완성할 TODO | 지정된 부분만 수정하고 파일 저장 |
| build 셀 | 소스를 컴파일하고 링크 | 첫 오류부터 읽고 수정 후 다시 실행 |
| learner verification | 자신의 코드가 계약을 만족하는지 검사 | 정답 확인 시점은 강사가 조절 |
| 접힌 solution | 한 가지 참고 구현 | 통과 후 자신의 코드와 다른 이유를 비교 |

이 노트북의 커널은 Python입니다. Python 셀이 C++ 파일을 `g++` 또는 CMake에 전달하고 결과를 화면에 보여 줍니다. 마지막의 `CalledProcessError`만 읽지 말고, 그 위에 나온 컴파일러 또는 CTest의 **첫 번째 오류** 를 먼저 찾으세요. 공통 용어와 문제 해결 절차는 `03.C++/BEGINNER_GUIDE.md`에 더 자세히 정리되어 있습니다.

> **구분: 수업 후 복습·자습**

## 이번 과정에서의 위치

**자습 확장 / 약 60분, 본수업 4시간에 포함하지 않음** . Python 커널로 실제 g++/CMake를 실행합니다. 먼저 환경 셀을 실행하세요.
원격 사용자도 GUI 없이 동일하게 진행합니다. 소스 저장 셀은 명시한 exercise 파일을 덮어쓰므로 노트북과 외부 편집기에서 동시에 수정하지 마세요.
미완성 TODO는 빌드에 성공해도 학습자 검증에 실패할 수 있습니다. 자신의 검증→접힌 정답→정답 검증 순서입니다.
빌드 결과는 `/workspace/Data/build/03.C++`에 보존됩니다. 컨테이너 재생성 후에는 환경 셀부터 실행합니다.


> **구분: 수업 후 복습·자습**

## 오늘의 60분 지도

| 시간 | 주제 | 도착점 |
|---:|---|---|
| 0–5분 | 환경·안전 규칙 | sanitizer의 역할 확인 |
| 5–17분 | 주소와 포인터 | `&`, `*`, `nullptr` 읽기 |
| 17–27분 | 참조와 배열 | 별칭과 포인터+길이 계약 구분 |
| 27–40분 | 동적 할당 | `new/delete`, `new[]/delete[]` 짝 이해 |
| 40–48분 | 수명 오류 | 누수, dangling, double delete 설명 |
| 48–57분 | 종합 TODO | 동적 센서 버퍼 완성·검증 |
| 57–60분 | 해답·점검 | 현재 절의 RAII 권장 방식 정리 |

> **구분: 수업 후 복습·자습**

## 학습 목표

1. 주소 연산자와 역참조 연산자를 구분한다.
2. 포인터와 참조의 null 가능성·재지정 가능성을 비교한다.
3. 동적 배열의 크기를 포인터와 별도로 전달해야 하는 이유를 설명한다.
4. 모든 `new`/`new[]`에 정확한 해제 연산을 짝짓는다.
5. AddressSanitizer와 UndefinedBehaviorSanitizer로 메모리 오류를 검사한다.

실습 파일은 [04_pointers_memory/exercise/main.cpp](04_pointers_memory/exercise/main.cpp)입니다. 아래 `%%writefile` 셀에서 source를 수정·저장하며, 위험한 코드를 일부러 추가하지 않아도 핵심을 익힐 수 있습니다.

In [ ]:
# 구분: 수업 후 복습·자습 셀
from pathlib import Path
import os
import shlex
import subprocess

def find_course_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / '03.C++']
    for parent in cwd.parents:
        candidates.extend([parent, parent / '03.C++'])
    for candidate in candidates:
        if (candidate / 'README.md').is_file() and (candidate / '04_pointers_memory').is_dir():
            return candidate
    raise FileNotFoundError('03.C++ 과정 폴더를 찾지 못했습니다.')

def run(args, cwd=None):
    command = [str(arg) for arg in args]
    print('$', shlex.join(command))
    result = subprocess.run(command, cwd=cwd or COURSE_DIR, text=True, capture_output=True, timeout=120)
    print(result.stdout, end='')
    if result.stderr: print(result.stderr, end='')
    result.check_returncode()
    return result

COURSE_DIR = find_course_dir()
LESSON_DIR = COURSE_DIR / '04_pointers_memory'
BUILD_DIR = COURSE_DIR.parent / 'Data' / 'build' / '03.C++' / LESSON_DIR.name
BUILD_DIR.mkdir(parents=True, exist_ok=True)

def compile_cpp(source, output_name, sanitize=False):
    output = BUILD_DIR / output_name
    flags = ['-std=c++17', '-Wall', '-Wextra', '-Wpedantic', '-O0', '-g']
    if sanitize:
        flags += ['-fsanitize=address,undefined', '-fno-sanitize-recover=undefined', '-fno-omit-frame-pointer']
    run(['g++', *flags, source, '-o', output])
    return output

print('교시 폴더:', LESSON_DIR)
run(['g++', '--version'])
os.chdir(COURSE_DIR.parent)

import sys
print('Python:', sys.executable)


> **구분: 수업 후 복습·자습**

## 1. 값, 주소, 포인터

```cpp
int score{80};
int* score_ptr{&score};

std::cout << score;       // 값 80
std::cout << score_ptr;   // score가 놓인 주소
std::cout << *score_ptr;  // 주소를 따라가 읽은 값 80
*score_ptr = 95;          // 같은 score를 변경
```

선언에서 `int*`는 'int를 가리키는 포인터 타입', 식에서 `&score`는 '주소 얻기', 식에서 `*score_ptr`은 '주소가 가리키는 객체에 접근하기(역참조)'입니다. 기호는 같아도 문맥에 따라 의미가 달라집니다.

가리킬 대상이 없음을 표현할 때는 숫자 `0`이나 `NULL`보다 타입 안전한 `nullptr`를 사용합니다. 역참조 전에 null이 아닌지 확인하세요.

> **구분: 수업 후 복습·자습**

### 포인터에도 타입과 `const`가 있다

```cpp
const int* pointer_to_const = &score;  // 포인터로 값을 변경할 수 없음
int* const const_pointer = &score;     // 다른 주소로 재지정할 수 없음
```

`const`가 `*`의 왼쪽이면 가리키는 값이 읽기 전용이고, 오른쪽이면 포인터 변수 자체가 고정됩니다. `int* first, second;`에서 포인터는 `first`뿐이므로 한 줄에 포인터 하나씩 선언하면 오해가 줄어듭니다.

> **구분: 수업 후 복습·자습**

### 주소 그림을 손으로 그리는 방법

포인터 문제를 머릿속 숫자로만 풀지 말고 객체 상자와 화살표를 그립니다.

```text
이름        저장 내용                 메모리 관계
number      [ 10 ]                    실제 int 객체
pointer     [ number의 주소 ] ───────> number
reference   (number의 다른 이름) ────> number
```

`pointer` 자체도 주소를 저장하는 별도 변수이므로 `&pointer`는 pointer 변수의 주소입니다. `pointer`는 number의 주소이고, `*pointer`는 number의 값입니다. 세 표현을 섞지 않도록 각 식 옆에 '주소' 또는 '값'을 적어 보세요.

포인터를 역참조하기 전에는 세 질문을 확인합니다: null이 아닌가, 가리키는 객체의 수명이 아직 끝나지 않았는가, 배열이라면 접근 인덱스가 전달된 길이보다 작은가. 이 세 조건 중 하나라도 모르면 읽거나 쓰지 않습니다.

> **구분: 수업 후 복습·자습**

## 2. 참조: null이 아닌 기존 객체의 별칭

```cpp
int score{80};
int& score_ref{score};
score_ref = 95;
```

| 속성 | 포인터 `T*` | 참조 `T&` |
|---|---|---|
| 대상 없음 | `nullptr` 가능 | 정상적인 참조는 항상 객체를 가리킴 |
| 다른 대상 가리키기 | 재지정 가능 | 초기화 후 다른 별칭으로 바뀌지 않음 |
| 접근 문법 | `*ptr`, `ptr->member` | 일반 변수처럼 `ref`, `ref.member` |
| 흔한 용도 | 선택적 대상, 배열 시작, 동적 자원 | 함수 매개변수, 반드시 존재하는 별칭 |

참조도 대상보다 오래 살 수는 없습니다. 지역 변수의 참조를 반환하면 포인터와 마찬가지로 dangling 상태가 됩니다.

> **구분: 수업 후 복습·자습**

### 45초 예측

```cpp
int value{10};
int* ptr{&value};
int& ref{value};
*ptr += 5;
ref *= 2;
```

마지막 `value`, `*ptr`, `ref`는 각각 얼마일까요? 셋은 서로 다른 값이 아니라 같은 객체를 세 가지 표현으로 보고 있습니다.

> **구분: 수업 후 복습·자습**

## 3. 배열과 포인터에는 길이가 함께 필요하다

```cpp
double sum(const double* values, std::size_t count) {
    double result{0.0};
    for (std::size_t i{0}; i < count; ++i) {
        result += values[i];
    }
    return result;
}
```

함수 매개변수에서 원시 배열은 보통 첫 원소를 가리키는 포인터로 전달되어 길이 정보가 사라집니다. 따라서 포인터와 `count`를 하나의 계약으로 다뤄야 합니다. 둘이 어긋나면 범위를 넘어 읽을 수 있습니다.

실무에서는 길이를 함께 보존하는 `std::vector`, `std::array`, C++20의 `std::span`이 더 안전합니다. 이번 과정은 C++17이므로 원리를 배우기 위해 `pointer + count`를 사용합니다.

> **구분: 수업 후 복습·자습**

## 4. 자동 수명과 동적 수명

<!-- procedure:lesson-08-9d235c:start -->
![메모리의 생성부터 해제까지](../교재그림/절차/lesson-08.svg)

포인터 값이 남아 있어도 유효한 메모리라는 뜻은 아닙니다. 생성·사용·해제 시점을 함께 추적합니다.
<!-- procedure:lesson-08-9d235c:end -->


블록 안의 일반 지역 변수는 범위를 벗어나면 자동으로 정리됩니다. `new`로 만든 객체는 명시적으로 해제하기 전까지 살아 있으므로 소유자가 책임져야 합니다.

```cpp
int* one{new int{42}};
delete one;
one = nullptr;

double* many{new double[count]{}};
delete[] many;
many = nullptr;
```

`new` ↔ `delete`, `new[]` ↔ `delete[]`를 정확히 짝지어야 합니다. `{}`를 붙인 동적 배열은 원소를 값 초기화합니다. 해제 후 `nullptr`로 바꾸면 같은 포인터를 우연히 다시 역참조할 가능성은 줄지만, 이미 복사해 둔 다른 포인터까지 안전해지는 것은 아닙니다.

> **구분: 수업 후 복습·자습**

## 5. 세 가지 대표 수명 오류

- **memory leak:** 소유 포인터를 잃어 `delete`할 수 없게 됨
- **dangling pointer:** 대상이 사라졌는데 주소만 남아 있음
- **double delete / use-after-free:** 이미 해제한 메모리를 다시 해제하거나 사용

다음은 설명을 위한 잘못된 코드이며 **실행하지 마세요.**

```cpp
int* value = new int{3};
delete value;
std::cout << *value;  // undefined behavior: 이미 수명이 끝남
delete value;         // undefined behavior: 두 번째 해제
```

Undefined behavior(정의되지 않은 동작)는 반드시 즉시 오류가 나는 것이 아닙니다. 우연히 정상처럼 보일 수도 있어 더 위험합니다. 그래서 컴파일러 경고와 sanitizer로 오류를 찾고, 실무에서는 현재 절에서 설명한 RAII 컨테이너로 소유권을 관리합니다.

### 저장 공간과 객체 수명은 다르다

포인터는 주소 비트를 저장하지만, 그 주소에 **살아 있는 객체가 있는지** 까지 기록하지 않습니다. `delete[]` 뒤에도 포인터 변수에는 예전 주소 숫자가 남을 수 있으나 배열 객체의 수명은 이미 끝났으므로 읽기·쓰기·두 번째 해제는 모두 유효하지 않습니다. `nullptr` 대입은 이 포인터 하나의 상태를 분명히 할 뿐, 같은 주소를 복사해 둔 다른 별칭을 고쳐 주지는 않습니다. 따라서 안전성을 판단할 때는 `null인가?`뿐 아니라 `누가 소유하는가?`, `대상이 아직 살아 있는가?`, `접근 길이가 맞는가?`를 함께 확인해야 합니다.

> **구분: 수업 후 복습·자습**

### RAII가 직접 `new`/`delete`보다 안전한 이유

RAII(Resource Acquisition Is Initialization)는 객체가 자원을 얻고, 그 객체의 수명이 끝날 때 소멸자가 자원을 정리하게 만드는 C++의 핵심 설계 방식입니다. `std::vector`는 내부 배열을 얻고 자신이 사라질 때 자동으로 해제하므로 중간 `return`이나 예외가 있어도 정리 코드가 빠지기 어렵습니다.

```text
직접 관리: new[] → 여러 분기와 return → 모든 경로에서 delete[]를 사람이 보장
RAII:       vector 생성 → 여러 분기와 return → 범위 종료 시 vector가 자동 정리
```

이 자습에서 `new[]`와 `delete[]`를 직접 쓰는 목적은 소유권과 수명을 눈으로 확인하기 위해서입니다. 실제 기능을 구현할 때는 먼저 `std::vector`, 고정 길이면 `std::array`, 단일 동적 객체의 독점 소유가 필요하면 `std::unique_ptr`를 검토하세요.

> **구분: 수업 후 복습·자습**

## 6. sanitizer로 검사하기

이 노트북은 메모리 실습에 다음 옵션을 추가합니다.

```bash
-fsanitize=address,undefined -fno-omit-frame-pointer -g
```

AddressSanitizer(ASan)는 범위 밖 접근, use-after-free, 일부 누수를 찾고, UndefinedBehaviorSanitizer(UBSan)는 여러 정의되지 않은 연산을 잡습니다. 검사가 조용히 끝났다는 것은 이번 실행에서 탐지된 문제가 없다는 뜻이지 모든 입력에 대한 완전한 증명은 아닙니다.

보고서가 나오면 첫 `ERROR:`와 첫 사용자 소스 위치부터 읽으세요.

> **구분: 수업 후 복습·자습**

### sanitizer 보고서를 읽는 순서

sanitizer가 문제를 찾으면 출력이 길어질 수 있습니다. 다음 네 부분을 위에서부터 찾습니다.

1. 첫 `ERROR: AddressSanitizer` 또는 `runtime error:`가 문제 종류를 알려 줍니다.
2. `READ of size ...` 또는 `WRITE of size ...`는 잘못된 읽기인지 쓰기인지 알려 줍니다.
3. stack trace에서 `/workspace/03.C++/...cpp:줄`처럼 자신의 소스가 처음 나오는 위치를 찾습니다.
4. 할당·해제 위치가 함께 나오면 같은 메모리의 수명 순서를 비교합니다.

보고서가 없다는 사실은 이번 입력에서 탐지된 문제가 없다는 뜻입니다. 모든 입력에서 안전하다는 수학적 증명은 아니므로 null, 빈 배열, 첫 인덱스, 마지막 유효 인덱스, 길이와 같은 인덱스를 각각 생각해야 합니다.

> **구분: 수업 후 복습·자습**

## 7. 안전한 예제 빌드·실행

각 소스에서 포인터가 가리키는 대상의 수명을 표시하고, `new[]`와 `delete[]`가 한 번씩 짝을 이루는지 확인한 뒤 실행하세요. 예제도 sanitizer를 켜고 빌드합니다.

> **구분: 수업 후 복습·자습**

### 예제 코드 해설 1 — `01_pointer_and_reference.cpp`: 같은 객체를 빌리는 두 방법

이 파일은 `<iostream>` 하나만 사용하며, 자동 수명을 가진 정수 `number`를 포인터와 참조가 각각 **빌려서** 변경하는 흐름을 보여 줍니다. `number`가 실제 객체이고 `pointer`와 `reference`는 별도의 정수 복사본이 아니라 같은 객체로 들어가는 두 접근 경로입니다. 둘 다 소유자가 아니므로 `delete`할 책임도 권한도 없습니다. 세 변수는 모두 `main()`의 지역 범위 안에 있어 `number`가 살아 있는 동안 포인터와 참조도 유효합니다.

| 실행 지점 | `number`의 값 | 코드가 표현하는 계약 |
|---|---:|---|
| `int number{10}` 직후 | 10 | 실제 저장 공간과 값을 만듦 |
| `increment(pointer)` 직후 | 11 | `int*`는 대상이 없을 수 있으므로 함수가 null을 검사한 뒤 `++(*value)`로 변경 |
| `double_value(reference)` 직후 | 22 | `int&`는 존재하는 객체를 받는 계약이므로 일반 변수처럼 곧바로 `value *= 2` |
| `*read_only_pointer` 출력 | 22 | `const int*`를 통해 같은 값을 읽되 이 포인터로는 수정할 수 없음 |

`increment(int* value)`에서 `value != nullptr` 검사가 **역참조보다 먼저** 실행되는 점이 핵심입니다. `++(*value)`의 괄호는 포인터가 가리키는 정수를 증가시킨다는 뜻을 분명히 합니다. 반면 `double_value(int& value)`의 참조 매개변수는 호출자가 반드시 살아 있는 정수를 제공해야 한다는 계약을 타입에 담습니다. 마지막 `increment(nullptr)` 호출은 이 차이를 직접 확인하는 경계 사례로, 검사에서 걸러져 상태도 출력도 바뀌지 않습니다.

`const int* read_only_pointer`에서 `const`는 원래 `number` 자체가 영원히 불변이라는 뜻이 아니라 **그 포인터를 통한 쓰기만 금지** 한다는 뜻입니다. 실행 결과 `현재 값: 22`가 나오면 `number → pointer → reference`가 아니라, 세 이름이 같은 객체를 바라보며 두 함수가 순서대로 상태를 바꿨음을 뜻합니다. 흔한 실수는 null 검사를 빼거나, 대상이 사라진 뒤 포인터·참조를 계속 쓰거나, 빌린 포인터를 소유 포인터로 오해해 해제하는 것입니다. 이어지는 exercise의 계산 함수도 `const double*`를 빌릴 뿐이므로 null과 길이를 검사하되 메모리를 해제해서는 안 됩니다.

> **구분: 수업 후 복습·자습**

### 예제 코드 해설 2 — `02_dynamic_array.cpp`: 할당부터 해제까지의 소유권

`<cstddef>`는 배열 크기와 인덱스에 쓰는 부호 없는 타입 `std::size_t`를, `<iostream>`은 결과 출력을 제공합니다. `constexpr std::size_t count{4}`는 크기가 실행 중 바뀌지 않는다는 뜻이고, `new int[count]{2, 4, 6, 8}`은 힙에 정수 네 개를 만든 뒤 그 첫 원소의 주소를 `values`에 돌려줍니다. 이 시점부터 `values`는 단순 관찰자가 아니라 그 동적 배열을 정리해야 하는 **소유 포인터** 입니다.

```text
할당 직후: values ──소유──▶ [2][4][6][8], count = 4
합산 반복: index = 0, 1, 2, 3만 접근 → sum = 20
delete[] 후: 배열 수명 종료, values에는 아직 예전 주소 비트가 남을 수 있음
nullptr 대입 후: 이 변수는 더 이상 어떤 객체도 가리키지 않음
```

반복 조건 `index < count`는 유효 인덱스 `[0, count)`를 정확히 표현합니다. `values[index]`는 포인터 연산으로 쓰면 `*(values + index)`와 같은 의미이므로, 실제 길이보다 큰 `count`를 사용하면 포인터 문법이 막아 주지 못하고 범위를 벗어납니다. 네 원소의 합은 `20`이며 첫 출력은 `합계: 20`이어야 합니다.

배열을 `new[]`로 만들었기 때문에 반드시 `delete[] values`로 짝을 맞춥니다. 단일 객체용 `delete`를 쓰거나 두 번 해제하면 정의되지 않은 동작입니다. 해제 뒤 `values = nullptr`는 이미 끝난 배열을 이 변수로 우연히 다시 역참조하는 일을 줄여 주고, `std::boolalpha`는 마지막 비교 결과를 숫자 `1` 대신 `true`로 출력하게 합니다. 다만 복사해 둔 다른 포인터까지 null이 되지는 않으므로 해제된 배열의 모든 별칭은 더 이상 사용하면 안 됩니다.

아래 실행 셀은 두 `.cpp`를 이름순으로 읽어 먼저 소스를 보여 준 다음, 각각 별도 실행 파일로 컴파일하고 sanitizer를 켠 채 실행합니다. 오류가 없다는 것은 이 입력과 실행 경로에서 잘못된 접근이 탐지되지 않았다는 뜻입니다. 이어지는 exercise도 `main()`이 동적 배열을 할당·해제하고 계산 함수는 `pointer + count`를 잠시 빌리는 같은 소유권 구조를 사용합니다. 실제 코드에서는 수동 정리 누락을 피하고 길이까지 함께 보존하기 위해 대개 `std::vector`를 우선 선택합니다.

In [ ]:
# 구분: 수업 후 복습·자습 셀
example_sources = sorted((LESSON_DIR / 'examples').glob('*.cpp'))
assert example_sources, 'examples 폴더에 .cpp가 없습니다.'

for source in example_sources:
    print(f'\n=== {source.name} ===')
    print(source.read_text(encoding='utf-8'))
    binary = compile_cpp(source, source.stem, sanitize=True)
    run([binary])

> **구분: 수업 후 복습·자습**

## 자주 하는 실수

- 초기화하지 않은 포인터에는 의미 없는 주소가 들어 있을 수 있습니다. 바로 대상을 주거나 `nullptr`로 초기화하세요.
- null 검사 뒤에도 대상의 수명이 끝났다면 포인터는 안전하지 않습니다. null 여부와 유효한 수명은 다른 문제입니다.
- `delete`와 `delete[]`를 바꾸어 쓰면 정의되지 않은 동작입니다.
- 포인터만 함수에 넘기고 실제 길이보다 큰 `count`를 주면 범위를 넘습니다.
- 포인터를 값으로 복사하면 소유권이 복사되는지 단순 관찰자만 늘어나는지 코드만 보고 알기 어렵습니다.
- 동적 배열이 필요하다는 이유만으로 `new[]`를 쓰지 마세요. 대개 `std::vector`가 길이와 정리를 함께 관리합니다.

> **구분: 수업 후 복습·자습**

## 8. 종합 실습 — 동적 센서 버퍼 (약 12분)

[04_pointers_memory/exercise/main.cpp](04_pointers_memory/exercise/main.cpp)의 세 구현 TODO를 완성하고, 마지막 정리 코드를 설명해 보세요.

1. `nullptr` 또는 길이 0을 안전하게 처리하며 합계를 구합니다.
2. 유효한 범위 안에서 최솟값을 찾습니다.
3. `copy_readings` 함수로 고정 샘플 배열을 `new[]`로 만든 동적 배열에 복사합니다.
4. 안전한 starter 실행을 위해 제공된 `delete[]`와 `nullptr` 대입이 왜 필요한지 설명합니다.

이번에는 의도적으로 원시 동적 배열을 사용하지만, 계산 함수는 메모리를 해제하지 않습니다. 할당한 `main`이 소유자이고 해제 책임도 `main`에 있습니다.

> **구분: 수업 후 복습·자습**

### starter 코드 해설 1 — 포인터와 길이는 하나의 계약입니다

`samples`는 `main`의 범위 안에 놓인 길이 5의 고정 배열이고, `readings`는 `new double[reading_count]{}`가 확보한 동적 배열의 첫 원소 주소를 보관합니다. 끝의 `{}` 때문에 starter의 동적 배열은 우선 모두 `0.0`으로 초기화됩니다. 그러나 `double*` 자체에는 배열 길이 정보가 없으므로 계산 함수는 주소 `readings`와 길이 `count`를 **항상 한 쌍** 으로 받습니다. 유효한 비어 있지 않은 입력이라면 `readings`가 적어도 `count`개의 살아 있는 `double`을 가리킨다는 것이 호출자와 함수 사이의 계약입니다.

두 계산 함수의 `const double*`는 값을 읽기만 하는 **비소유 관찰 포인터** 입니다. 함수는 배열 원소를 바꾸거나 `delete[]`해서는 안 됩니다. 반대로 `main`은 할당 결과를 가진 소유자이므로 계산이 끝난 뒤 정확히 한 번 해제할 책임이 있습니다. 이 구분 덕분에 같은 버퍼를 합계 함수와 최솟값 함수가 차례로 안전하게 빌려 쓸 수 있습니다.

`nullptr`와 길이 0은 원소를 하나도 읽을 수 없는 경계입니다. 두 함수 모두 **첫 역참조 전에** 이 경우를 처리해 `0.0`을 반환해야 합니다. 특히 최솟값의 초깃값을 첫 원소에서 얻고 싶다면 경계 검사가 반드시 먼저입니다. 경계를 통과한 뒤에만 인덱스 `0`부터 `count - 1`까지 접근할 수 있습니다. 실제 배열보다 큰 `count`를 넘기는 오류까지 포인터가 스스로 알아내지는 못하므로, 호출자도 정확한 길이를 전달해야 합니다.

> **구분: 수업 후 복습·자습**

### 실습 목적과 TODO 구현 명세

- **실습 목적:** 원시 동적 배열의 소유자와 이를 빌려 읽는 함수의 책임을 분리하고 `pointer + count` 계약을 안전하게 구현합니다.
- **구현 항목:** 빈 입력을 처리하는 합계·최솟값 함수와 고정 배열을 동적 배열로 복사하는 `copy_readings` 함수를 완성합니다.
- **입력/출력:** `const double*`와 길이 또는 다섯 샘플을 입력으로 받아 합계·평균·최솟값을 출력합니다.
- **제약:** null/길이 0에서는 역참조하지 않고 `0.0`을 반환하며, 인덱스는 `[0, count)`만 사용하고 할당·해제는 `main`에서 정확히 한 번 수행합니다.
- **완료 기준:** 대표 결과 `101 / 20.2 / 18.5`, nullptr·길이 0·양수·음수 입력, 복사 결과, ASan/UBSan 무오류 종료를 모두 만족해야 합니다.

#### TODO별 상태와 완료 기준

| TODO | 입력과 출력 | 수정되는 상태 | 완료를 확인하는 기준 |
|---|---|---|---|
| 1 `sum_readings` | 포인터와 길이를 받아 합계 `double`을 반환 | 입력 배열은 변경하지 않음 | null 또는 길이 0이면 `0.0`, 복사된 다섯 샘플이면 `101.0` |
| 2 `minimum_reading` | 포인터와 길이를 받아 최솟값 `double`을 반환 | 입력 배열은 변경하지 않음 | 빈 입력이면 `0.0`, 다섯 샘플이면 `18.5`; 유효 범위 밖을 읽지 않음 |
| 3 `copy_readings` | 원본 포인터·대상 포인터·길이를 받아 같은 인덱스에 복사 | `readings`의 5개 원소가 `0.0`에서 샘플 값으로 바뀜 | 출력이 합계 `101`, 평균 `20.2`, 최솟값 `18.5`가 됨 |

starter는 임시 `return 0.0`과 `static_cast<void>(...)` 덕분에 컴파일되지만, `copy_readings`가 미구현이어서 첫 복사 검증의 `assert`에서 멈춥니다. 복사되지 않은 `readings[0]`은 `0.0`, 기대값은 `18.5`입니다. 이는 예상된 TODO 실패이며 sanitizer나 설치 오류로 해석하지 않습니다. 아직 정상 출력·뒤의 계약 검사·정리 코드까지 실행한 상태가 아닙니다. TODO 완성 후 다시 빌드하여 출력과 sanitizer 종료 결과를 확인하세요. `new[]`로 확보했으므로 `delete[]`로 짝을 맞춰야 하며, 해제 뒤 `nullptr` 대입은 이 변수로 해제된 메모리를 다시 가리키지 않게 합니다. 다만 다른 곳에 복사해 둔 주소까지 null로 바꾸지는 않으므로, 해제 후에는 어떤 별칭도 역참조하면 안 됩니다.

권장 순서는 (1) 두 함수의 빈 입력 경계를 먼저 정하고, (2) `copy_readings`를 완성해 실제 검증 데이터를 만든 뒤, (3) 합계, (4) 최솟값을 구현하고, (5) 출력 세 줄과 sanitizer 무오류 종료를 함께 확인하는 것입니다. 제공된 정리 코드는 삭제하거나 계산 함수 안으로 옮기지 마세요.

- **변경·부작용:** 명세에서 지목한 지역 변수와 소유 상태만 변경합니다. 실행은 stdout을 출력하고 컴파일러가 build artifact를 만들지만 별도 데이터 파일이나 GUI 상태는 변경하지 않습니다.
- **오류·경계:** 명시된 값·수명·소유권 경계를 지키고, 컴파일 오류·예외·assert/CTest 실패는 숨기지 않습니다. 실패한 build 또는 run은 0이 아닌 종료 코드로 드러나야 합니다.

**TODO 2의 초기값:** 유효한 비어 있지 않은 입력에서는 `readings[0]`을 현재 최솟값으로 두고 나머지와 비교합니다. 0으로 시작하면 모든 값이 양수인 입력에서 잘못된 최솟값이 됩니다.

**TODO 3의 완전한 계약:** `copy_readings(source, destination, count)`는 반환값이 없는 `void` 함수입니다. 어느 포인터든 null이거나 count가 0이면 아무것도 읽거나 쓰지 않고 끝냅니다. 그 밖에는 `0 <= index < count`에서 `destination[index]`를 `source[index]`와 같게 만들며 원본은 바꾸지 않습니다. 두 배열은 겹치지 않고 count개 이상 유효하다고 가정합니다. 검증은 `{1.25,2.0,7.5}`의 복사와 null·0 길이 무동작도 확인합니다.


> **구분: 수업 후 복습·자습**

### 노트북 편집 → 파일 저장 계약

아래 펼친 `%%writefile` 셀이 [03.C++/04_pointers_memory/exercise/main.cpp](04_pointers_memory/exercise/main.cpp)의 학습 원본입니다. 셀의 `TODO`를 수정한 뒤 실행하면 기존 파일을 덮어씁니다. 그 다음 build/run 셀을 실행하세요. 외부 편집기와 동시에 수정하지 말고, 노트북을 저장해 수정한 셀을 보존하세요.


> **구분: 수업 후 복습·자습 셀**


In [ ]:
%%writefile 03.C++/04_pointers_memory/exercise/main.cpp
#include <cassert>
#include <cmath>
#include <cstddef>
#include <iostream>

bool almost_equal(double actual, double expected) {
    constexpr double tolerance{1e-9};
    return std::abs(actual - expected) < tolerance;
}

double sum_readings(const double* readings, std::size_t count) {
    // TODO 1: nullptr와 빈 배열을 안전하게 처리하고 모든 값을 더하세요.
    static_cast<void>(readings);
    static_cast<void>(count);
    return 0.0;
}

double minimum_reading(const double* readings, std::size_t count) {
    // TODO 2: 가장 작은 값을 찾으세요. 입력이 비어 있으면 0.0을 반환합니다.
    static_cast<void>(readings);
    static_cast<void>(count);
    return 0.0;
}

void copy_readings(
    const double* source,
    double* destination,
    std::size_t count
) {
    // TODO 3: null을 안전하게 처리하고 같은 인덱스에 모든 값을 복사하세요.
    static_cast<void>(source);
    static_cast<void>(destination);
    static_cast<void>(count);
}

int main() {
    constexpr std::size_t reading_count{5};
    const double samples[reading_count]{18.5, 21.0, 19.5, 22.0, 20.0};
    double* readings{new double[reading_count]{}};

    copy_readings(samples, readings, reading_count);

    // 복사 결과와 함수 계약을 화면 문자열이 아닌 실제 메모리와 반환값으로 검사합니다.
    for (std::size_t index{0}; index < reading_count; ++index) {
        assert(almost_equal(readings[index], samples[index]));
    }
    const double negative_samples[3]{-4.5, -8.0, -2.25};
    assert(almost_equal(sum_readings(nullptr, reading_count), 0.0));
    assert(almost_equal(sum_readings(samples, 0), 0.0));
    assert(almost_equal(sum_readings(samples, reading_count), 101.0));
    assert(almost_equal(sum_readings(negative_samples, 3), -14.75));
    assert(almost_equal(minimum_reading(nullptr, reading_count), 0.0));
    assert(almost_equal(minimum_reading(samples, 0), 0.0));
    assert(almost_equal(minimum_reading(samples, reading_count), 18.5));
    assert(almost_equal(minimum_reading(negative_samples, 3), -8.0));

    const double sum{sum_readings(readings, reading_count)};
    const double average{sum / static_cast<double>(reading_count)};
    const double minimum{minimum_reading(readings, reading_count)};

    std::cout << "합계: " << sum << '\n';
    std::cout << "평균: " << average << '\n';
    std::cout << "최솟값: " << minimum << '\n';

    // 안전한 starter 실행을 위해 정리 코드는 제공합니다.
    // new[]와 delete[]가 짝이고, 해제 뒤 nullptr로 바꾸는 이유를 설명해 보세요.
    delete[] readings;
    readings = nullptr;
    assert(readings == nullptr);

    return 0;
}


In [ ]:
# 구분: 수업 후 복습·자습 셀
from hashlib import sha256
from pathlib import Path

active_sources = [
    '03.C++/04_pointers_memory/exercise/main.cpp'
]
for relative_path in active_sources:
    source_path = Path(relative_path)
    payload = source_path.read_bytes()
    print(f'{source_path}: sha256={sha256(payload).hexdigest()}')

exercise_source = COURSE_DIR.parent / active_sources[0]


In [ ]:
# 구분: 수업 후 복습·자습 셀
exercise_binary = compile_cpp(exercise_source, 'exercise_app', sanitize=True)


In [ ]:
# 구분: 수업 후 복습·자습 셀
learner_result = run([exercise_binary])
expected_output = "합계: 101\n평균: 20.2\n최솟값: 18.5\n"
assert learner_result.stdout == expected_output
memory_contract_binary = BUILD_DIR / "exercise_contract"
run([
    "g++", "-std=c++17", "-Wall", "-Wextra", "-Wpedantic", "-O0", "-g",
    "-fsanitize=address,undefined", "-fno-sanitize-recover=undefined", "-fno-omit-frame-pointer",
    f'-DMEMORY_SOURCE="{exercise_source}"',
    LESSON_DIR / "tests" / "check_memory.cpp", "-o", memory_contract_binary,
])
contract_result = run([memory_contract_binary])
assert contract_result.stdout == "memory function/copy boundary contract PASS\n"
print("복사 결과와 nullptr·빈·일반·음수 입력 assert를 모두 통과했습니다.")
print("sanitizer가 오류 없이 종료되었습니다.")
print("learner_verification=PASS")

> **구분: 수업 후 복습·자습**

### 기대 결과

```text
합계: 101
평균: 20.2
최솟값: 18.5
```

고정 assert와 별도 계약 프로그램은 두 함수의 nullptr·길이 0·일반·음수 입력과 `copy_readings`의 복사 결과를 모두 직접 검사합니다. sanitizer 보고서도 없어야 합니다. 실수 비교는 계산 오차를 고려해 참고 해답처럼 작은 허용 오차를 사용할 수 있습니다.
**sanitizer 통과의 의미:** 출력 숫자만 맞아서는 완료가 아닙니다. `-fno-sanitize-recover=undefined`는 UBSan이 정의되지 않은 연산을 발견하면 실패로 종료하게 합니다. `runtime error:` 또는 ASan의 `ERROR:`가 보이면 최초 발생 위치와 소유권·범위·수명을 수정한 뒤 다시 빌드합니다. 경고를 지우거나 검증 셀을 건너뛰지 않습니다.


> **구분: 수업 후 복습·자습**

<details>
<summary><strong>참고 해답 확인</strong></summary>

해답은 합계·평균·최솟값과 경계 입력을 `assert`로 확인하고, 해제 뒤 소유 포인터가 `nullptr`인지 검사합니다. 아래 실행은 학습자 파일을 변경하지 않습니다.
</details>

In [ ]:
# 구분: 수업 후 복습·자습 셀
# 참고 해답을 출력해 경계 검사와 유효 인덱스 순서를 비교합니다.
solution_source = LESSON_DIR / 'solution' / 'main.cpp'
print(solution_source.read_text(encoding='utf-8'))
# sanitizer를 켠 별도 실행 파일로 대표값과 null/빈 입력 assert를 검증합니다.
solution_binary = compile_cpp(solution_source, 'solution_app', sanitize=True)
solution_result = run([solution_binary])
print('참고 해답의 assert와 sanitizer 검사를 통과했습니다.')

### 정답 해설 — 입력부터 결과까지

**길이와 수명부터 검사:** 포인터만으로 배열 개수를 알 수 없으므로 주소와 길이를 함께 읽습니다. 유효한 index는 0부터 n-1이며 i==n은 배열 밖입니다. 값 전달은 복사본, 참조·포인터를 통한 쓰기는 원본에 영향을 줄 수 있습니다.

3칸 배열 `[10,20,30]`에서 마지막 원소는 index 2입니다. 동적 배열을 만든 소유자가 한 번 해제하고, 해제 이후 주소를 남겨 다시 읽으면 안 됩니다. 일반적인 소유는 vector 같은 자동 수명 관리 타입으로 하고 직접 할당은 원리 확인용입니다.

**확인 질문:** 포인터가 null이 아니라는 사실만으로 살아 있는 배열과 올바른 길이를 보장할 수 있나요? 정답의 검사와 해제 순서를 테스트 사례에 연결하세요. 이 절은 자습입니다.

**적용·재검증:** 자신의 답안을 저장한 뒤 지정된 exercise 파일/작성 셀에 반영하고 저장→exercise 빌드→학습자 검증 순서로 실행합니다. solution 빌드는 별도 참고 결과입니다. 오류는 첫 compiler/CTest 메시지부터 읽습니다. `.so`를 이미 로드했다면 재빌드 후 새 Python 프로세스 또는 커널 재시작과 환경 셀 재실행으로 다시 로드합니다. 결과는 기존 Data 경로에 저장합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](04_pointers_memory/solution/main.cpp)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```cpp
#include <cassert>
#include <cmath>
#include <cstddef>
#include <iostream>

bool almost_equal(double actual, double expected) {
    constexpr double tolerance{1e-9};
    return std::abs(actual - expected) < tolerance;
}

double sum_readings(const double* readings, std::size_t count) {
    if (readings == nullptr || count == 0) {
        return 0.0;
    }

    double sum{0.0};
    for (std::size_t index{0}; index < count; ++index) {
        sum += readings[index];
    }
    return sum;
}

double minimum_reading(const double* readings, std::size_t count) {
    if (readings == nullptr || count == 0) {
        return 0.0;
    }

    double minimum{readings[0]};
    for (std::size_t index{1}; index < count; ++index) {
        if (readings[index] < minimum) {
            minimum = readings[index];
        }
    }
    return minimum;
}

void copy_readings(
    const double* source,
    double* destination,
    std::size_t count
) {
    if (source == nullptr || destination == nullptr) {
        return;
    }
    for (std::size_t index{0}; index < count; ++index) {
        destination[index] = source[index];
    }
}

int main() {
    constexpr std::size_t reading_count{5};
    const double samples[reading_count]{18.5, 21.0, 19.5, 22.0, 20.0};
    double* readings{new double[reading_count]{}};

    copy_readings(samples, readings, reading_count);

    // 복사 결과와 함수 계약을 화면 문자열이 아닌 실제 메모리와 반환값으로 검사합니다.
    for (std::size_t index{0}; index < reading_count; ++index) {
        assert(almost_equal(readings[index], samples[index]));
    }
    const double negative_samples[3]{-4.5, -8.0, -2.25};
    assert(almost_equal(sum_readings(nullptr, reading_count), 0.0));
    assert(almost_equal(sum_readings(samples, 0), 0.0));
    assert(almost_equal(sum_readings(samples, reading_count), 101.0));
    assert(almost_equal(sum_readings(negative_samples, 3), -14.75));
    assert(almost_equal(minimum_reading(nullptr, reading_count), 0.0));
    assert(almost_equal(minimum_reading(samples, 0), 0.0));
    assert(almost_equal(minimum_reading(samples, reading_count), 18.5));
    assert(almost_equal(minimum_reading(negative_samples, 3), -8.0));

    const double sum{sum_readings(readings, reading_count)};
    const double average{sum / static_cast<double>(reading_count)};
    const double minimum{minimum_reading(readings, reading_count)};

    std::cout << "합계: " << sum << '\n';
    std::cout << "평균: " << average << '\n';
    std::cout << "최솟값: " << minimum << '\n';

    // 안전한 starter 실행을 위해 정리 코드는 제공합니다.
    // new[]와 delete[]가 짝이고, 해제 뒤 nullptr로 바꾸는 이유를 설명해 보세요.
    delete[] readings;
    readings = nullptr;
    assert(readings == nullptr);

    return 0;
}
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 후 복습·자습 셀
# 정답 코드 실행 검증
assert solution_result.returncode == 0
print('solution_verification=PASS')

> **구분: 수업 후 복습·자습**

## 권장 실무와 후속 자습: RAII

현재 코드는 `new[]`와 `delete[]` 사이에 예외나 조기 `return`이 추가되면 해제를 빠뜨릴 수 있습니다. C++의 핵심 해법은 자원의 수명을 객체 수명에 묶는 **RAII(Resource Acquisition Is Initialization)** 입니다.

- 여러 원소: 우선 `std::vector<double>`
- 단독 동적 객체의 단일 소유: `std::unique_ptr<T>`
- 정말 공유해야 하는 소유: `std::shared_ptr<T>`

실무와 후속 자습에서는 같은 소유권 문제를 명시적 `delete` 없이 해결하는 RAII 타입을 우선 사용합니다.

> **구분: 수업 후 복습·자습**

## 마무리 점검

1. 선언의 `*`, 식의 `&`, 식의 `*`는 각각 무엇을 뜻하나요?
2. 포인터와 참조는 대상 없음과 재지정 가능성에서 어떻게 다른가요?
3. 배열 포인터만으로 원소 개수를 알 수 없는 이유는 무엇인가요?
4. `new[]`에 `delete`를 쓰면 왜 안 되나요?
5. 누수, dangling pointer, double delete를 한 문장씩 설명할 수 있나요?

**한 줄 요약:** 포인터의 안전성은 주소값 하나가 아니라 **대상의 수명, 길이, 소유자** 를 함께 알 때 확보됩니다.